# **XGBoost** классификация


In [1]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [2]:
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep, append_results, eval_thresholds
from preprocessing.target import ttp_target, hybrid_target
from metrics.Metrics import merged_metrics
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

обучение 1 модели

In [3]:
from xgboost import XGBClassifier

def train_xgb_ttp(
    df,
    train_size,
    test_size,
    step,
):

    splitter = prep(
        df=df,
        target_fn=ttp_target,
        target_name="ttp",
        target_col="TTP_class",
        horizons=[12, 24, 48],
        train_size=train_size,
        test_size=test_size,
        step=step,
        target_kwargs={"n_classes": 3},
        scale_cols=[
            "Open", "High", "Low", "Close",
            "Alligator_Jaw", "Alligator_Teeth", "Alligator_Lips",
            "AO",
            "AddOn_Anchor_Level", "AddOn_Size_Pct"
        ]
    )

    for X_train, X_test, y_train, y_test, scaler in splitter:

        model = XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            random_state=42
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        metrics = merged_metrics(y_test, y_pred)

        append_results({
            "task_type": "classification",

            "model_name": "XGBoost",
            "model_family": "xgb",
            "model_params": {
                "n_estimators": 300,
                "max_depth": 5,
                "learning_rate": 0.05,
                "subsample": 0.8,
                "colsample_bytree": 0.8
            },

            "target_name": "ttp",
            "target_variant": "3class",
            "horizons": "12_24_48",

            **metrics
        })

train_xgb_ttp(
    df=df,
    train_size=1000,
    test_size=200,
    step=100
)


обучение 2 моделей: сначала TTP (precision), затем гибридная (precision)

In [ ]:
from xgboost import XGBClassifier

def train_xgb_ttp_precision_then_hybrid_precision(
    df,
    train_size,
    test_size,
    step,
):

    splitter = prep(
        df=df,
        target_fn=ttp_target,
        target_name="ttp",
        target_col="TTP_class",
        horizons=[12, 24, 48],
        train_size=train_size,
        test_size=test_size,
        step=step,
        target_kwargs={"n_classes": 3},
        extra_target_fns=[hybrid_target],
        scale_cols=[
            "Open", "High", "Low", "Close",
            "Alligator_Jaw", "Alligator_Teeth", "Alligator_Lips",
            "AO",
            "AddOn_Anchor_Level", "AddOn_Size_Pct"
        ]
    )

    NO_PROFIT_IDX = 2 

    for wfs_i, (X_train, X_test, y_ttp_train, y_ttp_test, scaler) in enumerate(splitter):

        
        ttp_model = XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            random_state=42
        )

        ttp_model.fit(X_train, y_ttp_train)
        ttp_pred_test = ttp_model.predict(X_test)

        append_results({
            "task_type": "classification",
            "model_name": "XGBoost",
            "model_family": "xgb",
            "target_name": "ttp",
            "target_variant": "3class",
            "optimization_goal": "precision_no_profit",
            "wfs_step": wfs_i,
            **merged_metrics(y_ttp_test, ttp_pred_test)
        })

        
        y_h_train = X_train["GoodTrade"]
        y_h_test  = X_test["GoodTrade"]

        X_train_h = X_train.drop(columns=["GoodTrade"]).copy()
        X_test_h  = X_test.drop(columns=["GoodTrade"]).copy()

        proba_train = ttp_model.predict_proba(X_train)
        proba_test  = ttp_model.predict_proba(X_test)

        X_train_h["TTP_no_profit_proba"] = proba_train[:, NO_PROFIT_IDX]
        X_test_h["TTP_no_profit_proba"]  = proba_test[:, NO_PROFIT_IDX]

        hybrid_model = XGBClassifier(
            objective="binary:logistic",
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.7,
            colsample_bytree=0.7,
            tree_method="hist",
            random_state=42
        )

        hybrid_model.fit(X_train_h, y_h_train)

        hybrid_proba = hybrid_model.predict_proba(X_test_h)[:, 1]

        
        rows = eval_thresholds(y_h_test, hybrid_proba)

        for r in rows:
            append_results({
                "task_type": "classification",
                "model_name": "XGBoost",
                "model_family": "xgb",
                "target_name": "hybrid",
                "target_variant": "GoodTrade",
                "optimization_goal": "precision_goodtrade_soft",
                "threshold": r["threshold"],
                "coverage": r["coverage"],
                "precision_goodtrade": r["precision"],
                "recall_goodtrade": r["recall"],
                "wfs_step": wfs_i,
            })

In [5]:
train_xgb_ttp_precision_then_hybrid_precision(
    df=df,
    train_size=1000,
    test_size=200,          
    step=100
)
